In [1]:
# Cell 1: Install necessary libraries
# We need 'transformers' for the models and 'scikit-learn' for the classification report
!pip install transformers scikit-learn

In [2]:
# Cell 2: Imports and Configuration

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from tqdm.auto import tqdm # For nice progress bars
import os

# --- Configuration ---
# We'll use BioBERT as our baseline model
MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.1'

# File names
TRAIN_FILE = 'healthver_train.csv'
DEV_FILE = 'healthver_dev.csv'

# Training parameters
BATCH_SIZE = 8       # Batch size. 8 or 16 is good for BERT-base models on Colab GPUs.
NUM_EPOCHS = 3       # How many times to train on the full dataset. 3 is a good starting point.
LEARNING_RATE = 2e-5 # The recommended learning rate for fine-tuning BERT.
MAX_LENGTH = 512     # Max sequence length. 512 is the max for BERT.

# Where to save the final model
SAVE_PATH = './biobert_baseline'

In [3]:
# Cell 3: Load Data and Prepare Labels

print("Loading data...")
try:
    df_train = pd.read_csv(TRAIN_FILE)
    df_dev = pd.read_csv(DEV_FILE)
    print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: Could not find {TRAIN_FILE} or {DEV_FILE}.")
    print("Please make sure you have uploaded all 3 healthver_*.csv files to your Colab session.")

# We need to turn the text labels into numbers.
# We 'fit' the encoder on the training data to learn the mapping.
label_encoder = LabelEncoder()
label_encoder.fit(df_train['label'])

# Store the class names for our final report
LABELS = label_encoder.classes_
print(f"Labels have been encoded. Mapping:")
print(list(zip(range(len(LABELS)), LABELS)))

Loading data...
Data loaded successfully.
Labels have been encoded. Mapping:
[(0, 'Neutral'), (1, 'Refutes'), (2, 'Supports')]


In [4]:
# Cell 4: Define the PyTorch Dataset

class HealthVerDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label_encoder, max_length):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.max_length = max_length
        self.label_encoder = label_encoder

        # Combine claim and evidence into a single string for BERT
        self.texts = (self.data['claim'] + " [SEP] " + self.data['evidence']).tolist()

        # Encode the labels using the 'fitted' encoder from Cell 3
        self.labels = self.label_encoder.transform(self.data['label'])

    def __len__(self):
        # How many items are in the dataset
        return len(self.texts)

    def __getitem__(self, idx):
        # Get one item (text and label)
        text = self.texts[idx]
        label = self.labels[idx]

        # Tokenize the combined text
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,    # Adds [CLS] and [SEP]
            max_length=self.max_length, # Pad or truncate to MAX_LENGTH
            padding='max_length',       # Pad to max_length
            truncation=True,            # Truncate if longer than max_length
            return_attention_mask=True, # We need this mask
            return_tensors='pt'         # Return PyTorch tensors
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [5]:
# Cell 5: Initialize Model, Dataloaders, and Optimizer

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Creating datasets and dataloaders...")
# Create the dataset objects
train_dataset = HealthVerDataset(df_train, tokenizer, label_encoder, MAX_LENGTH)
dev_dataset = HealthVerDataset(df_dev, tokenizer, label_encoder, MAX_LENGTH)

# Create the dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_dataloader = DataLoader(dev_dataset, batch_size=BATCH_SIZE) # No shuffle for dev

print("Loading model...")
# Set the device to GPU (cuda) if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the model. We tell it we have 3 labels.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABELS))
model.to(device) # Move the model to the GPU

# Set up the optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# --- THIS IS THE BASELINE ---
# We use the standard loss function with NO class weights.
loss_fct = CrossEntropyLoss()

# Set up the learning rate scheduler
num_training_steps = NUM_EPOCHS * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

print("Setup complete. Ready for training.")

Loading tokenizer for dmis-lab/biobert-base-cased-v1.1...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Creating datasets and dataloaders...
Loading model...
Using device: cuda


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Setup complete. Ready for training.


In [6]:
# Cell 6: The Training Loop

print(f"Starting training for {NUM_EPOCHS} epochs...")

# Loop over each epoch
for epoch in range(NUM_EPOCHS):
    model.train()  # Set the model to training mode
    total_loss = 0

    # Use tqdm for a progress bar
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    # Loop over each batch of data
    for batch in progress_bar:
        # Move the batch to the GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Clear old gradients
        model.zero_grad()

        # Forward pass (run data through the model)
        # We pass 'labels' so the model computes the loss for us
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

        # Get the loss
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass (calculate gradients)
        loss.backward()

        # Update weights
        optimizer.step()

        # Update learning rate
        lr_scheduler.step()

        # Update the progress bar description
        progress_bar.set_postfix({'loss': loss.item()})

    # Print average loss for the epoch
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"\nEpoch {epoch+1} average training loss: {avg_train_loss:.4f}")

print("--- Training complete ---")

Starting training for 3 epochs...


Epoch 1/3:   0%|          | 0/1324 [00:00<?, ?it/s]


Epoch 1 average training loss: 0.6649


Epoch 2/3:   0%|          | 0/1324 [00:00<?, ?it/s]


Epoch 2 average training loss: 0.2611


Epoch 3/3:   0%|          | 0/1324 [00:00<?, ?it/s]


Epoch 3 average training loss: 0.1226
--- Training complete ---


In [7]:
# Cell 7: The Evaluation Loop

print("Starting evaluation on the development (dev) set...")
model.eval()  # Set the model to evaluation mode (disables dropout, etc.)

all_preds = [] # To store model predictions
all_true = []  # To store true labels

# We don't need to calculate gradients here, so we wrap in 'torch.no_grad()'
with torch.no_grad():
    # Loop over the dev dataloader
    for batch in tqdm(dev_dataloader, desc="Evaluating"):
        # Move batch to GPU
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask)

        # Get the model's predictions (the class with the highest logit score)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)

        # Move predictions and labels back to CPU and store them
        all_preds.extend(predictions.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

print("--- Evaluation complete ---")

Starting evaluation on the development (dev) set...


Evaluating:   0%|          | 0/240 [00:00<?, ?it/s]

--- Evaluation complete ---


In [8]:
# Cell 8: Show Results and Save Model

print("\n--- BASELINE MODEL CLASSIFICATION REPORT (Dev Set) ---")
print("This report is for the model trained on the original, IMBALANCED data.\n")

# Generate and print the report
report = classification_report(
    all_true,
    all_preds,
    target_names=LABELS, # Use the label names we stored in Cell 3
    digits=4
)
print(report)

print("--------------------------------------------------")
print("PAY ATTENTION TO THIS: ")
print("Find the row for 'Refutes' and look at its 'f1-score'.")
print("This is your BASELINE F1-SCORE that we will try to beat in the next phase.")
print("--------------------------------------------------")


# --- Save the Model ---
if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

print(f"Saving baseline model to {SAVE_PATH}...")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Model saved. You can now download this folder from the Colab file panel.")


--- BASELINE MODEL CLASSIFICATION REPORT (Dev Set) ---
This report is for the model trained on the original, IMBALANCED data.

              precision    recall  f1-score   support

     Neutral     0.8888    0.8046    0.8446       993
     Refutes     0.6262    0.6726    0.6486       391
    Supports     0.6472    0.7261    0.6844       533

    accuracy                         0.7559      1917
   macro avg     0.7207    0.7344    0.7258      1917
weighted avg     0.7680    0.7559    0.7601      1917

--------------------------------------------------
PAY ATTENTION TO THIS: 
Find the row for 'Refutes' and look at its 'f1-score'.
This is your BASELINE F1-SCORE that we will try to beat in the next phase.
--------------------------------------------------
Saving baseline model to ./biobert_baseline...
Model saved. You can now download this folder from the Colab file panel.


Now we See that the ovrerall Accuracy is around 76 percent and f-score for refutes class is the least.So we work to improve our baseline model by adjusting the class weights by increaing the weight for the refutes class.Refer to the Next weights Updated BioBert Model notebook for that results.